# 1 Hrv

Time- and frequency-domain HRV linked to psychometric question timing.

**Reads:** `data/case-study/ (one participant, all sessions)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/case-study/processed'
PSY = '../data/case-study/psychometric'

sessions = [1, 2, 3]
colors = ['b', 'g', 'r']
question_type = 'HADS'

# load all sessions
hrv_data = {}
psy_data = {}

for s in sessions:
    hrv = pd.read_csv(f'{DATA}/hrv_{s:02d}.csv')
    hrv['datetime'] = pd.to_datetime(hrv['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    hrv_data[s] = hrv.sort_values('datetime')

    psy = pd.read_csv(f'{PSY}/Psychometric_Test_Results_{s:02d}.csv')
    psy['Question Start Time'] = pd.to_datetime(psy['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    psy['Question Answer Time'] = pd.to_datetime(psy['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
    psy_data[s] = psy

# interval average HRV per question
session_merged = {}
for s in sessions:
    data = psy_data[s][psy_data[s]['Type'] == question_type].reset_index(drop=True).copy()
    hrv = hrv_data[s]

    # interval average, not point estimate
    for metric in ['sdnn', 'rmssd']:
        data[metric] = [
            hrv.loc[
                (hrv['datetime'] >= row['Question Start Time']) &
                (hrv['datetime'] <= row['Question Answer Time']),
                metric
            ].mean()
            for _, row in data.iterrows()
        ]

    data['Question Number'] = data['Test'].str.extract(r'(\d+)').astype(int)
    data['Start Time Difference'] = (data['Question Start Time'] - data['Question Start Time'].min()).dt.total_seconds()
    data['Answer Time Difference'] = (data['Question Answer Time'] - data['Question Start Time'].min()).dt.total_seconds()
    session_merged[s] = data

# plot SDNN
plt.figure(figsize=(14, 7))
for s, color in zip(sessions, colors):
    m = session_merged[s]
    plt.plot(m['Start Time Difference'], m['sdnn'], label=f'SDNN - {question_type} Session {s}', marker='o', color=color)
    plt.plot(m['Answer Time Difference'], m['sdnn'], linestyle='--', marker='x', color=color)
    for i in range(len(m)):
        plt.annotate(m['Question Number'].iloc[i].item(), (m['Start Time Difference'].iloc[i], m['sdnn'].iloc[i]))

plt.title(f'SDNN During {question_type} Questions')
plt.xlabel('Time Difference (seconds)')
plt.ylabel('SDNN (ms)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
plt.close()

# plot RMSSD
plt.figure(figsize=(14, 7))
for s, color in zip(sessions, colors):
    m = session_merged[s]
    plt.plot(m['Start Time Difference'], m['rmssd'], label=f'RMSSD - {question_type} Session {s}', marker='o', color=color)
    plt.plot(m['Answer Time Difference'], m['rmssd'], linestyle='--', marker='x', color=color)
    for i in range(len(m)):
        plt.annotate(m['Question Number'].iloc[i].item(), (m['Start Time Difference'].iloc[i], m['rmssd'].iloc[i]))

plt.title(f'RMSSD During {question_type} Questions')
plt.xlabel('Time Difference (seconds)')
plt.ylabel('RMSSD (ms)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import lombscargle
import matplotlib.pyplot as plt

DATA = '../data/case-study/processed'

# load IBI data
ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

def frequency_hrv(ibi_values):
    # artifact rejection
    ibi = ibi_values[(ibi_values > 300) & (ibi_values < 2000)].values
    if len(ibi) < 10:
        return None
    
    # cumulative time axis
    t = np.cumsum(ibi) / 1000.0
    t = t - t[0]
    
    # detrend
    nn = ibi - np.mean(ibi)
    
    # frequency range
    freqs = np.linspace(0.01, 0.5, 500)
    angular_freqs = 2 * np.pi * freqs
    
    # Lomb-Scargle PSD
    psd = lombscargle(t, nn, angular_freqs, normalize=False)
    psd = psd / len(nn)
    
    # frequency bands
    vlf_mask = (freqs >= 0.003) & (freqs < 0.04)
    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs < 0.4)
    
    vlf_power = np.trapezoid(psd[vlf_mask], freqs[vlf_mask])
    lf_power = np.trapezoid(psd[lf_mask], freqs[lf_mask])
    hf_power = np.trapezoid(psd[hf_mask], freqs[hf_mask])
    total_power = vlf_power + lf_power + hf_power
    lf_hf_ratio = lf_power / hf_power if hf_power > 0 else np.nan
    
    return {
        'VLF': vlf_power, 'LF': lf_power, 'HF': hf_power,
        'Total': total_power, 'LF/HF': lf_hf_ratio,
        'LF%': lf_power / total_power * 100 if total_power > 0 else 0,
        'HF%': hf_power / total_power * 100 if total_power > 0 else 0,
        'freqs': freqs, 'psd': psd
    }

sessions = {
    'Baseline': ibi_baseline,
    'Session 01': ibi_01,
    'Session 02': ibi_02,
    'Session 03': ibi_03
}

# compute frequency HRV
results = {}
for name, df in sessions.items():
    r = frequency_hrv(df['ibi'].dropna())
    if r:
        results[name] = r

# summary table
print("=== Frequency-Domain HRV ===\n")
print(f"{'Session':<14} {'VLF':>8} {'LF':>10} {'HF':>10} {'LF/HF':>8} {'LF%':>6} {'HF%':>6}")
for name, r in results.items():
    print(f"{name:<14} {r['VLF']:>8.1f} {r['LF']:>10.1f} {r['HF']:>10.1f} {r['LF/HF']:>8.2f} {r['LF%']:>5.1f}% {r['HF%']:>5.1f}%")

# clinical interpretation
print("\n=== Clinical Interpretation ===\n")
print("LF/HF > 2.0 = sympathetic dominance (stress/anxiety)")
print("LF/HF < 0.5 = parasympathetic dominance (relaxation)")
print("LF/HF 0.5-2.0 = balanced autonomic tone\n")
for name, r in results.items():
    ratio = r['LF/HF']
    state = "sympathetic dominant" if ratio > 2.0 else "parasympathetic dominant" if ratio < 0.5 else "balanced"
    print(f"{name}: LF/HF={ratio:.2f} -> {state}")

# PSD plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'Baseline': 'gray', 'Session 01': 'tab:blue', 'Session 02': 'tab:orange', 'Session 03': 'tab:green'}

for ax, (name, r) in zip(axes.flat, results.items()):
    freqs, psd = r['freqs'], r['psd']
    ax.plot(freqs, psd, color=colors.get(name, 'black'), linewidth=0.8)
    ax.fill_between(freqs, psd, where=(freqs >= 0.04) & (freqs < 0.15), alpha=0.3, color='red', label='LF')
    ax.fill_between(freqs, psd, where=(freqs >= 0.15) & (freqs < 0.4), alpha=0.3, color='blue', label='HF')
    ax.set_title(f'{name} (LF/HF={r["LF/HF"]:.2f})')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (ms²/Hz)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 0.5)

plt.suptitle('Frequency-Domain HRV: Power Spectral Density')
plt.tight_layout()
plt.show()
plt.close()

# LF/HF bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
ratios = [results[n]['LF/HF'] for n in names]
bars = ax.bar(names, ratios, color=[colors[n] for n in names])
ax.axhline(2.0, color='red', linestyle='--', alpha=0.7, label='Stress threshold')
ax.axhline(0.5, color='green', linestyle='--', alpha=0.7, label='Relaxation threshold')
ax.set_ylabel('LF/HF Ratio')
ax.set_title('Sympathovagal Balance Across Sessions')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()